# Notebook 06: Comprehensive Evaluation

This notebook runs the comprehensive evaluation suite for the Sepsis-Associated AKI Early Warning System.

## Evaluation Metrics Explained
- **DCA (Decision Curve Analysis):** Evaluates the clinical utility of a prediction model by calculating the net benefit at different threshold probabilities. It helps answer if using the model leads to better clinical decisions than 'treat all' or 'treat none'.
- **ECE (Expected Calibration Error):** Measures the discrepancy between predicted probabilities and actual observed outcomes. A lower ECE means the model's confidence scores are highly reliable.

In [ ]:
import os
import sys
from pathlib import Path

project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
from src import evaluation, models, features

print("Loading test split...")
test_df = pd.read_parquet('../data/processed/test.parquet')

print("Loading trained models...")
stage1, stage2 = models.load_models('../models')

s1_feats = features.get_stage1_features()
s2_feats = features.get_all_features(test_df)
print(f"test: {test_df.shape}  |  stage1 horizons {list(stage1)}  |  stage2 horizons {list(stage2)}")

In [ ]:
print("Running full evaluation suite for Stage-2 models @ 24h (LGBM vs XGB)...")

H = 24
models_24h = {
    'stage2_lgbm_24h': stage2[H]['lgbm']['model'],
    'stage2_xgb_24h':  stage2[H]['xgb']['model'],
}
X_test = test_df[s2_feats]
y_test = test_df[f'aki_label_{H}h'].values

# Example subgroup: elderly vs adult (creatinine baselines differ by age)
subgroups = (test_df['Age'] >= 65).map({True: 'elderly (65+)', False: 'adult (<65)'})

results = evaluation.run_full_evaluation(
    models_24h, X_test, y_test, subgroups=subgroups, output_dir='../outputs'
)
print("\nModel comparison @ 24h:")
display(results['comparison'])

# Per-horizon discrimination: Stage-1 screener vs Stage-2 full model
print("\nPer-horizon AUROC / AUPRC (test set):")
for h in [6, 12, 24]:
    yt = test_df[f'aki_label_{h}h'].values
    m1 = evaluation.evaluate_model(yt, stage1[h]['model'].predict_proba(test_df[s1_feats])[:, 1])
    m2 = evaluation.evaluate_model(yt, stage2[h]['lgbm']['model'].predict_proba(test_df[s2_feats])[:, 1])
    print(f"  {h:>2}h | Stage1 AUROC={m1['auroc']:.3f} AUPRC={m1['auprc']:.3f}"
          f"   |  Stage2 AUROC={m2['auroc']:.3f} AUPRC={m2['auprc']:.3f}")

print("\nPlots saved to ../outputs/figures/ ; tables to ../outputs/reports/")

## Reading the Results

- **Discrimination (AUROC / AUPRC):** how well each model ranks at-risk patients. On the 100-patient demo these will be optimistic/noisy — that is expected and should be stated plainly in the demo. The point is that the full evaluation suite runs end-to-end.
- **Calibration (ECE / reliability curve):** whether predicted probabilities match observed AKI rates — critical before any clinical thresholding. Isotonic calibration was fit on the validation split during training.
- **Decision Curve Analysis (DCA):** net clinical benefit versus "treat all" / "treat none" across threshold probabilities.
- **Subgroup analysis:** performance split by age band — a first check for the kind of population-shift failure that broke the Epic Sepsis Model.

> ⚠️ Demo caveat: with ~20 test patients these numbers are illustrative, not validation-grade. Real validation requires the full MIMIC-IV cohort plus an external set (eICU-CRD) — see the README roadmap.